# 01 — Explore connectome (Phase 1)

Sanity-check the FlyBrain graph builder on the synthetic source and on whatever Zenodo CSVs you point it at. Plots focus on:

1. **Degree distribution** — log-log of out-degree before / after compression.
2. **Modularity Q** — directed Newman-Leicht for each compression method.
3. **Region / cell-type distribution** — counts per super-node after aggregation.

The notebook runs end-to-end with only `numpy` + the FlyBrain wheel (no plotting deps required, but matplotlib is used if available).

In [ ]:
from __future__ import annotations

import collections
from pathlib import Path

import numpy as np

from flybrain.graph import (
    FlyGraph,
    build_synthetic,
    compress,
    compress_and_aggregate,
)

try:
    import matplotlib.pyplot as plt

    HAS_MPL = True
except ImportError:
    HAS_MPL = False

OUT_DIR = Path("data/flybrain/notebook")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(42)

## 1 — Build a synthetic source graph

The synthetic generator is deterministic for a given (`num_nodes`, `seed`). It produces a small fly-inspired graph with four regions (`MB`, `AL`, `OL`, `CX`) and eight cell types, intended as a stand-in for the FlyWire connectome in CI / smoke tests.

In [ ]:
G = build_synthetic(num_nodes=2048, seed=42)
print(f"nodes={G.num_nodes:,}  edges={G.num_edges:,}")
print("provenance:", G.provenance)
regions = collections.Counter(n.region for n in G.nodes)
celltypes = collections.Counter(n.node_type for n in G.nodes)
print("regions:", dict(regions))
print("cell types:", dict(celltypes))

## 2 — Degree distribution

In [ ]:
def out_degrees(g: FlyGraph) -> np.ndarray:
    deg = np.zeros(g.num_nodes, dtype=np.int64)
    for s, _t in g.edge_index:
        deg[s] += 1
    return deg


deg = out_degrees(G)
print(f"degree min={deg.min()} median={int(np.median(deg))} max={deg.max()} mean={deg.mean():.1f}")
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(deg, bins=40, log=True, color="steelblue")
    ax.set_xlabel("out-degree")
    ax.set_ylabel("# nodes (log)")
    ax.set_title("Synthetic source — degree distribution")
    fig.tight_layout()

## 3 — Modularity across compression methods

Compute directed modularity `Q` for each compression method at `K=64`. Higher is better — Louvain / Leiden should dominate, region/celltype aggregation is a (deterministic) baseline.

In [ ]:
rows: list[tuple[str, int, float]] = []
for method in ["region_agg", "celltype_agg", "louvain", "leiden", "spectral"]:
    a = compress(G, method=method, target_k=64, seed=42)
    rows.append((method, a.num_clusters, a.modularity))
    print(f"{method:>13}  K={a.num_clusters:3d}  Q={a.modularity:+.4f}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(5, 3))
    names = [r[0] for r in rows]
    qvals = [r[2] for r in rows]
    ax.barh(names, qvals, color="darkorange")
    ax.axvline(0, color="black", lw=0.5)
    ax.set_xlabel("directed modularity Q")
    fig.tight_layout()

## 4 — Aggregated graph for K∈{32, 64, 128, 256}

Compress + collapse with Louvain. Inspect compressed-graph stats and the regional / cell-type composition of each super-node.

In [ ]:
for k in [32, 64, 128, 256]:
    Gk = compress_and_aggregate(G, method="louvain", target_k=k, seed=42)
    deg_k = out_degrees(Gk)
    print(
        f"K={k:3d}  edges={Gk.num_edges:5d}  "
        f"deg(median={int(np.median(deg_k))}, max={deg_k.max()})  "
        f"compression_ratio={G.num_edges / max(Gk.num_edges, 1):6.1f}x"
    )

## 5 — Region distribution per super-node (K=64, Louvain)

In [ ]:
G64 = compress_and_aggregate(G, method="louvain", target_k=64, seed=42)
region_counts = collections.Counter(n.region for n in G64.nodes)
print("super-node region counts:")
for r, c in region_counts.most_common():
    print(f"  {r:6s} {c}")

# Cluster sizes from provenance.
sizes: list[int] = list(G64.provenance.get("cluster_sizes", [])) or []
if sizes:
    arr = np.asarray(sizes)
    print(f"cluster sizes: min={arr.min()} median={int(np.median(arr))} max={arr.max()}")

## 6 — (Optional) load a real FlyWire / Zenodo CSV pair

If you have downloaded the connectome export, point `ZENODO_DIR` to a directory with `neurons.csv` and `connections.csv` and uncomment.

In [ ]:
# from flybrain.graph import load_zenodo
# Z = load_zenodo(dir="data/flywire/raw")
# print(f"FlyWire: nodes={Z.num_nodes:,} edges={Z.num_edges:,}")
# print("provenance:", Z.provenance)